# Notebook 2 — Document Visual QA

**Hands-on synthetic data pipeline workshop with NeMo Data Designer and NeMo Anonymizer — PyData London 2026**

## What you'll build

By the end of this notebook you'll have a small **multimodal visual QA dataset** grounded in synthetic business document images with charts, tables, KPI cards, annotations, and dense text. This follows the iterative synthetic-data generation approach used for long-document VLM training: start from visual seeds, generate targeted QA, judge visual grounding, then keep the rows that pass. See the [iterative SDG story](https://nvidia-nemo.github.io/DataDesigner/latest/devnotes/training-a-vlm-to-understand-long-documents-an-iterative-sdg-story/) for the full pipeline pattern.

A finished row will look roughly like this:

| png_base64 | primary_visual | secondary_visual | layout_style | document_condition | question_type | question_difficulty | visual_focus | question | answer | judge | failure_review |
|---|---|---|---|---|---|---|---|---|---|---|---|
| `<base64 PNG>` | stacked area chart showing category mix over six months | KPI card row with current value, target, delta, and traffic-light status | executive dashboard page with three horizontal bands | clean digital render | short answer | medium | `{ "focus_area": "chart_reading", "available_focus_areas": ["chart_reading", "kpi_status", "cross_panel_reasoning"], "rationale": "The page has a stacked area chart with labeled month-by-month category shares." }` | `{ "text": "Which category has the largest share in the final month of the stacked area chart?", "rationale": "The question targets the stacked area chart's final-month category shares." }` | `Enterprise has the largest share in the final month.` | `{ "passes": true, "reason": "The answer is visually grounded in the chart and directly answers the question." }` | `N/A` |

Rows that fail the judge also get a `failure_review` note so you can debug prompt or seed issues. Passing rows set `failure_review` to `N/A`.

## What you'll learn

1. **Image inputs into LLM columns** — `multi_modal_context` lets a VLM see the document when generating the question and the answer.
2. **The same pipeline shape, applied to images** — sample → seed → generate → judge → filter. The framework barely changes; the *content* does.
3. **Visual focus classification** — a VLM first classifies which focus areas the page can support, using the image plus minimal seed metadata as orientation.
4. **Structured question generation** — Pydantic schemas give each question typed fields: `text` and `rationale`.
5. **Multimodal judging** — the judge looks at the image too, so it can verify visual grounding, not just text plausibility.
6. **Preview → Create loop** — iterate on a few visual rows, inspect the image/question/answer side by side, then scale up only when you're happy.

## Pipeline shape

```
Rich document images (seed)
        │
        ▼
   ┌────────────────────────────────┐
   │ question_type / difficulty     │  sampler columns (no LLM)
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ visual_focus                   │  VLM structured column (Pydantic)
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ question                       │  VLM structured column (Pydantic)
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ answer                         │  VLM text column
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ judge (passes + reason)        │  VLM structured column
   └────────────────────────────────┘
        │
        ▼
   ┌────────────────────────────────┐
   │ failure_review                 │  VLM text column (N/A for passing rows)
   └────────────────────────────────┘
        │
        ▼
    filter to passing rows
```

## Note on time and cost

VLM inference is slower than text. Keep `num_records` small in the live workshop (3 for preview, 10–15 for create).

## Setup

In [ ]:
from notebook_helpers import (
    ARTIFACT_DIR,
    DATA_DIR,
    display_base64_image,
    display_image_with_qa,
    environment_setup,
    load_document_seed,
    show_provider_info,
)
from IPython.display import display
from prompts import (
    VQA_ANSWER_PROMPT,
    VQA_FAILURE_REVIEW_PROMPT,
    VQA_JUDGE_PROMPT,
    VQA_QUESTION_PROMPT,
    VQA_VISUAL_FOCUS_PROMPT,
)
from pydantic import BaseModel, Field

# Pick which hosted backend to use. Leave as "auto" to use whichever key is in
# your .env (precedence: NVIDIA -> OpenRouter -> OpenAI), or set explicitly to
# "nvidia", "openrouter", or "openai" to flip between providers without editing
# any other files.
PROVIDER = "openai"

provider = environment_setup(provider=PROVIDER)
show_provider_info(provider)

## Part 1 — Inspect the seed data

The seed dataset is a checked-in parquet at `data/rich_document_seed.parquet` containing
synthetic business document images as base64. Each row has a `png_base64` column plus
four orientation fields: `primary_visual`, `secondary_visual`, `layout_style`, and `document_condition`.

The documents intentionally contain **rich visual information** — chart labels,
table rows, KPI deltas, owner names, dates, annotations, and source notes — so
the VLM has to reason over more than plain text extraction.

The `visual_focus` prompt receives the image and the four orientation fields,
then classifies which focus area the row can support before question generation.

In [ ]:
seed = load_document_seed()
print(f"Loaded {len(seed)} document image(s).\n")
seed.head()

In [ ]:
display_base64_image(seed.iloc[0]["png_base64"])

### Seed path for Data Designer

The parquet already contains a `png_base64` column -- no extra encoding step needed.

In [ ]:
seed_path = str(DATA_DIR / "rich_document_seed.parquet")
avg_kb = seed["png_base64"].str.len().mean() / 1024
print(f"Seed: {seed_path}  ({len(seed)} rows, avg {avg_kb:.0f} KB base64 per image)")

## Part 2 — Build the VLM pipeline

Same builder, same column types as Notebook 1 — but the LLM columns gain
`multi_modal_context=[ImageContext(...)]` so the VLM sees the document page when
it generates. The `ImageContext` references the `png_base64` column in
the seed parquet — the VLM endpoint is remote, so base64 is the only
reliable way to pass image bytes. Before writing a question, the pipeline runs
a structured `visual_focus` classifier so the question prompt targets visual
evidence the page actually contains.

The pipeline includes a **multimodal judge** that also sees the image. A text-only
judge can score plausibility but can't verify visual grounding — passing the same
image to the judge lets it check whether the answer actually matches what's visible.
The judge returns a boolean `passes` verdict and a short reason. Under the hood,
the rubric checks two things:

- **answer_correctness** — does the answer match what's on the page?
- **visual_grounding** — does the question require examining visual structure?

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner, ResumeMode

config_builder = dd.DataDesignerConfigBuilder(model_configs=provider.model_configs)
data_designer = DataDesigner(model_providers=provider.model_providers)
data_designer.set_run_config(dd.RunConfig(
    disable_early_shutdown=True,
))

config_builder.with_seed_dataset(
    seed_source=dd.LocalFileSeedSource(path=seed_path),
    sampling_strategy=dd.SamplingStrategy.ORDERED,
)

In [ ]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="question_type",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "multiple choice",
                "yes or no",
                "short answer",
                "list of items",
            ],
            weights=[0.2, 0.2, 0.4, 0.2],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="question_difficulty",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["easy", "medium", "hard"],
            weights=[0.25, 0.35, 0.40],
        ),
    )
)


In [ ]:
IMAGE_CONTEXT = [
    dd.ImageContext(
        column_name="png_base64",
        data_type=dd.ModalityDataType.BASE64,
        image_format=dd.ImageFormat.PNG,
    )
]

class VisualFocus(BaseModel):
    focus_area: str = Field(description="The single best focus-area label for this document page.")
    available_focus_areas: list[str] = Field(description="All focus-area labels clearly supported by visible page content.")
    rationale: str = Field(description="One sentence explaining the visible evidence behind the chosen focus area.")


config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="visual_focus",
        prompt=VQA_VISUAL_FOCUS_PROMPT,
        model_alias=provider.vlm_alias,
        output_format=VisualFocus,
        multi_modal_context=IMAGE_CONTEXT,
    )
)


class VQAQuestion(BaseModel):
    text: str = Field(description="The question text (include A-D options on separate lines for multiple choice).")
    rationale: str = Field(description="One sentence explaining which visual element on the page this question targets.")


config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="question",
        prompt=VQA_QUESTION_PROMPT,
        model_alias=provider.vlm_alias,
        output_format=VQAQuestion,
        multi_modal_context=IMAGE_CONTEXT,
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="answer",
        prompt=VQA_ANSWER_PROMPT,
        model_alias=provider.vlm_alias,
        multi_modal_context=IMAGE_CONTEXT,
        extract_reasoning_content=True,
    )
)

class QAJudge(BaseModel):
    passes: bool = Field(description="True if the QA pair is correct, visually grounded, and usable for training.")
    reason: str = Field(description="One sentence justifying the verdict.")


config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="judge",
        prompt=VQA_JUDGE_PROMPT,
        model_alias=provider.vlm_alias,
        output_format=QAJudge,
        multi_modal_context=IMAGE_CONTEXT,
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="failure_review",
        prompt=VQA_FAILURE_REVIEW_PROMPT,
        model_alias=provider.vlm_alias,
        multi_modal_context=IMAGE_CONTEXT,
        skip=dd.SkipConfig(when="{{ judge.passes }}", value="N/A"),
    )
)

data_designer.validate(config_builder)
print("Visual QA pipeline config validated.")

### Preview (includes judge)

The preview runs question, answer, and judge. If the judge fails a row,
`failure_review` runs as a debugging column; passed rows set it to `N/A`. VLM calls are
slow; expect 5-30 seconds per row. Keep `num_records` low here.

In [ ]:
preview = data_designer.preview(config_builder, num_records=3)

In [ ]:
preview.display_sample_record()

In [ ]:
row = preview.dataset.iloc[2]
q = row["question"]
vf = row["visual_focus"]
question_text = q["text"] if isinstance(q, dict) else str(q)
focus_area = vf.get("focus_area", str(vf)) if isinstance(vf, dict) else str(vf)
display_image_with_qa(
    png_base64=row["png_base64"],
    question=question_text,
    answer=row["answer"],
    label=f"preview: {focus_area}",
    failure_review=row.get("failure_review", "N/A"),
    show_failure_review=True,
)


## Part 3 — Scale up, filter, save

Smaller `num_records` than Notebook 1 — VLM calls are expensive and we want the workshop to fit in time. We'll keep the live run small, then filter the generated rows by the multimodal judge.

The structured `visual_focus` column classifies what the page can support before
the question is generated. This keeps focus selection explicit instead of hiding
fallback logic inside the question prompt.

In [ ]:
results = data_designer.create(
    config_builder,
    num_records=10,
    dataset_name="02_document_visual_qa",
)
df = results.load_dataset()
print(f"Generated {len(df)} rows.")
display(df[["question_type", "question_difficulty", "visual_focus", "question", "answer", "judge", "failure_review"]].head())

In [ ]:
if len(df):
    review_row = df.iloc[0]
    q = review_row["question"]
    vf = review_row["visual_focus"]
    question_text = q["text"] if isinstance(q, dict) else str(q)
    focus_area = vf.get("focus_area", str(vf)) if isinstance(vf, dict) else str(vf)
    display_image_with_qa(
        png_base64=review_row["png_base64"],
        question=question_text,
        answer=review_row["answer"],
        label=f"generated: {focus_area}",
        failure_review=review_row.get("failure_review", "N/A"),
        show_failure_review=True,
    )

In [ ]:
def keep(row):
    judge = row["judge"]
    if isinstance(judge, dict):
        return bool(judge.get("passes", False))
    return False

keep_mask = df.apply(keep, axis=1)
filtered = df[keep_mask].reset_index(drop=True)
print(f"Kept {len(filtered)} / {len(df)} rows ({len(filtered) / len(df):.0%} keep rate).")

# Optional local export for later inspection.
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
visual_qa_path = ARTIFACT_DIR / "02_visual_qa.parquet"
filtered.to_parquet(visual_qa_path, index=False)
print(f"Saved {len(filtered)} rows -> {visual_qa_path.relative_to(ARTIFACT_DIR.parent)}")

### Resume an interrupted run

Data Designer can resume an interrupted `create()` call when the saved run is compatible with the current config. Keep the same `dataset_name`, keep the pipeline settings stable, and pass `resume=ResumeMode.IF_POSSIBLE`.

Use `ResumeMode.ALWAYS` when you want an incompatible saved run to fail loudly instead of starting fresh.


In [ ]:
# Optional: resume an interrupted run. Leave this off during the live workshop unless your run stopped midway.
RESUME_INTERRUPTED_RUN = False

if RESUME_INTERRUPTED_RUN:
    resumed_results = data_designer.create(
        config_builder,
        num_records=10,
        dataset_name="02_document_visual_qa",
        resume=ResumeMode.IF_POSSIBLE,
    )
    resumed_df = resumed_results.load_dataset()
    print(f"Resumed/generated {len(resumed_df)} rows.")
else:
    print("Set RESUME_INTERRUPTED_RUN = True to resume an interrupted run.")


### Partition larger runs

Resume helps when one run is interrupted. Partitioning helps when a larger job should be split across separate workers or notebook sessions. Each worker builds the same pipeline, but uses `PartitionBlock(index=partition_index, num_partitions=NUM_PARTITIONS)` to claim one slice of the seed dataset and writes to a partition-specific `dataset_name`.

The sketch below is a `run_partitions.py`-style main program. It creates independent jobs with non-overlapping source records, plus a clear path to concatenate the parquet outputs afterward.


In [ ]:
# Example main program for run_partitions.py. Do not run this during the live workshop.
from concurrent.futures import ProcessPoolExecutor, as_completed

RUN_PARALLEL_PARTITIONS = False
NUM_PARTITIONS = 4
RECORDS_PER_PARTITION = 250
MAX_WORKERS = 4
DATASET_PREFIX = "02_document_visual_qa"


def add_visual_qa_columns(builder, partition_provider):
    builder.add_column(
        dd.SamplerColumnConfig(
            name="question_type",
            sampler_type=dd.SamplerType.CATEGORY,
            params=dd.CategorySamplerParams(
                values=[
                    "multiple choice",
                    "yes or no",
                    "short answer",
                    "list of items",
                ],
                weights=[0.2, 0.2, 0.4, 0.2],
            ),
        )
    )
    builder.add_column(
        dd.SamplerColumnConfig(
            name="question_difficulty",
            sampler_type=dd.SamplerType.CATEGORY,
            params=dd.CategorySamplerParams(
                values=["easy", "medium", "hard"],
                weights=[0.25, 0.35, 0.40],
            ),
        )
    )
    image_context = [
        dd.ImageContext(
            column_name="png_base64",
            data_type=dd.ModalityDataType.BASE64,
            image_format=dd.ImageFormat.PNG,
        )
    ]

    builder.add_column(
        dd.LLMStructuredColumnConfig(
            name="visual_focus",
            prompt=VQA_VISUAL_FOCUS_PROMPT,
            model_alias=partition_provider.vlm_alias,
            output_format=VisualFocus,
            multi_modal_context=image_context,
        )
    )

    builder.add_column(
        dd.LLMStructuredColumnConfig(
            name="question",
            prompt=VQA_QUESTION_PROMPT,
            model_alias=partition_provider.vlm_alias,
            output_format=VQAQuestion,
            multi_modal_context=image_context,
        )
    )
    builder.add_column(
        dd.LLMTextColumnConfig(
            name="answer",
            prompt=VQA_ANSWER_PROMPT,
            model_alias=partition_provider.vlm_alias,
            multi_modal_context=image_context,
            extract_reasoning_content=True,
        )
    )
    builder.add_column(
        dd.LLMStructuredColumnConfig(
            name="judge",
            prompt=VQA_JUDGE_PROMPT,
            model_alias=partition_provider.vlm_alias,
            output_format=QAJudge,
            multi_modal_context=image_context,
        )
    )
    builder.add_column(
        dd.LLMTextColumnConfig(
            name="failure_review",
            prompt=VQA_FAILURE_REVIEW_PROMPT,
            model_alias=partition_provider.vlm_alias,
            multi_modal_context=image_context,
            skip=dd.SkipConfig(when="{{ judge.passes }}", value="N/A"),
        )
    )
    return builder


def build_partition_builder(partition_index):
    partition_provider = environment_setup(provider=PROVIDER)
    builder = dd.DataDesignerConfigBuilder(model_configs=partition_provider.model_configs)
    builder.with_seed_dataset(
        seed_source=dd.LocalFileSeedSource(path=seed_path),
        sampling_strategy=dd.SamplingStrategy.ORDERED,
        selection_strategy=dd.PartitionBlock(
            index=partition_index,
            num_partitions=NUM_PARTITIONS,
        ),
    )
    add_visual_qa_columns(builder, partition_provider)
    return partition_provider.model_providers, builder


def run_partition(partition_index):
    partition_model_providers, builder = build_partition_builder(partition_index)
    partition_designer = DataDesigner(model_providers=partition_model_providers)
    partition_designer.set_run_config(dd.RunConfig(disable_early_shutdown=True))
    results = partition_designer.create(
        builder,
        num_records=RECORDS_PER_PARTITION,
        dataset_name=f"{DATASET_PREFIX}_part_{partition_index:02d}_of_{NUM_PARTITIONS:02d}",
    )
    return {
        "partition": partition_index,
        "dataset_path": str(results.artifact_storage.final_dataset_path),
    }


if RUN_PARALLEL_PARTITIONS:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(run_partition, partition_index): partition_index
            for partition_index in range(NUM_PARTITIONS)
        }
        for future in as_completed(futures):
            print(future.result())
else:
    print("Set RUN_PARALLEL_PARTITIONS = True in a standalone script to launch partitioned jobs.")


## What just happened

Identical pipeline to Notebook 1 — but the seed is images, the model is a VLM, and the judge sees the image. The framework didn't change. *Adding multimodal didn't add a new architecture; it added a `multi_modal_context` argument.*

## Next

Look at the answers your pipeline generated. Notice how often the useful signal depends on visual structure: chart labels, table cells, KPI deltas, highlights, handwritten notes, and panel layout. That is the point of multimodal synthetic data — the answer should be grounded in what the model can see, not only in plain text.

[`03_privacy_with_anonymizer.ipynb`](03_privacy_with_anonymizer.ipynb) — switch to production-style usage data and apply replacement strategies that preserve utility while reducing leakage risk.